Groundwater | Transport Modeling Track

# Step 4: Model Implementation — Building the Spill→Capture GWT Model

> **The 10 steps at a glance:** 1-Goal → 2-Perceptual → 3-MODFLOW → **4-Build** → 5-Calibrate → 6-Validate → 7-Sensitivity → 8-Apply → 9-Document → 10-Communicate

**Story so far:** In [03t](03t_modflow_transport.ipynb) you translated the perceptual model into MODFLOW 6 language — the advection-dispersion-reaction equation, the Péclet/Courant accuracy criteria, and the GWT packages (with **SRC** as the source for a known-mass spill). Now we **build and run** the model: a contaminant spills upgradient of a GWHE doublet, and we read the breakthrough at the extraction (monitoring) well.

## The question — and the trap

A contaminant **spills upgradient** of a GWHE doublet running flow-only (clean injection, extraction). The extraction well is the **monitoring / compliance well**, and the question is blunt: **does the plume reach the well above the regulatory threshold, and when — or bypass it?**

Here is the trap. The plume in the source→well corridor is **narrow** — only as wide as transverse dispersion lets it spread ($\alpha_T = 1$ m). The inherited flow grid has ~50 m cells; when a cell is far larger than the spreading length, the scheme adds **numerical dispersion** that smears the plume — most damagingly **sideways** — and mis-reads the concentration along the centre-line the well draws from. So we **refine the source→well corridor** to ~10 m, which resolves the *longitudinal* breakthrough; the *transverse* width stays under-resolved (we return to that in §5).

> **A demo, not a build-it-yourself.** We call a small helper — `transport_srcpulse_demo` — that assembles the coupled steady-flow / transient-transport model for us (a **known-mass SRC pulse** on the refined corridor), so this notebook can focus on *reading and judging* the result.

## Timing and Learning Objectives

| | |
|---|---|
| **Time** | ~20–30 min |
| **Prerequisites** | Transport Steps 1–3 |

By the end you will be able to:

1. **Build and run** a coupled GWF/GWT spill→capture model with a finite **SRC** mass pulse
2. Read the **breakthrough** at the monitoring well and state a **threshold verdict** (reach? exceed? when?)
3. Check the model with a **mass balance** and a **solubility** guardrail
4. Interpret **Péclet/Courant** as *accuracy* criteria, and know which claims the grid supports (→ particle tracking for the lateral question)

In [ ]:
# Setup
import sys
sys.path.insert(0, '../../_SUPPORT/src')
sys.path.insert(0, '../../_SUPPORT/src/scripts/scripts_exercises')

import numpy as np
import matplotlib.pyplot as plt

from transport_srcpulse_demo import build_srcpulse_demo
from shared_functions import check_task_with_solution, create_multiple_choice

---
## 1 — Build the model

The helper builds a **coupled steady-GWF / transient-GWT** model on a corridor-refined grid: the doublet runs **flow-only** (clean injection $+Q$, extraction $-Q$), and the spill enters as a **finite-pulse SRC mass loading** — grams released over the pulse duration, adding no water. The first call builds and solves (~13 s); later calls reuse the cache.

In [ ]:
# Stated pedagogical regulatory threshold (mg/L = g/m3)
THRESHOLD_mgL = 3.0

# Build + run the spill->capture SRC-pulse demo (cached after the first run)
res = build_srcpulse_demo(mass_g=3.0e5, pulse_days=30.0, total_days=120.0, solubility_mgL=1000.0)

print(f"Doublet (flow-only)  : injection cell {res.inj_cell}, extraction/monitoring cell {res.ext_cell}")
print(f"Spill (SRC pulse)    : {res.mass_g:.0f} g over {res.pulse_days:.0f} d  ->  {res.smassrate_gpd:.0f} g/d")
print(f"  spill location     : ({res.spill_xy[0]:.0f}, {res.spill_xy[1]:.0f})  (upgradient of the extraction well)")
print(f"Grid Peclet          : Pe_L {res.PeL_min:.1f}-{res.PeL_max:.1f} (<= 2 OK)   Pe_T {res.PeT_min:.0f}-{res.PeT_max:.0f} (>> 2)")

---
## 2 — Numerical accuracy: Péclet and Courant

`Pe ≤ 2` (cell size `≤ 2·α`) and `Cr ≤ 1` (time step `≤ Δx/v`) are **accuracy** criteria — with TVD the solution stays *stable* above them, but it **smears**. Refining the corridor to ~10 m brings the **longitudinal** `Pe_L` under 2, so the well breakthrough is trustworthy. The **transverse** `Pe_T` stays far above 2 at any feasible cell size (because `α_T = 1` m) — the limitation we return to in §5.

In [ ]:
print("Accuracy criteria (not stability -- TVD stays stable above them, but smears):")
print(f"  Longitudinal grid Peclet  Pe_L = {res.PeL_min:.1f}-{res.PeL_max:.1f}   (target <= 2  -> resolved on the refined corridor)")
print(f"  Transverse   grid Peclet  Pe_T = {res.PeT_min:.0f}-{res.PeT_max:.0f}    (target <= 2  -> NOT resolved; alpha_T = 1 m is too small for any feasible cell)")
print()
print("The corridor is refined to ~10 m so Pe_L <= 2: the LONGITUDINAL breakthrough at the")
print("well is trustworthy. The TRANSVERSE axis stays under-resolved at any affordable grid.")

### Checkpoint 1 — grid Péclet

In [ ]:
check_task_with_solution('task_t04_checkpoint_1')

---
## 3 — Result: breakthrough at the monitoring well

Read the concentration at the extraction (monitoring) well over time, against the stated regulatory threshold: **does it reach the well, does it exceed the threshold, and when?**

In [ ]:
t, c = res.times, res.breakthrough
peak, t_peak = res.peak_mgL, res.arrival_day
exceed = c > THRESHOLD_mgL
t_first = float(t[np.argmax(exceed)]) if exceed.any() else None

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(t, c, color='firebrick', lw=2.2, label='concentration at the monitoring well')
ax.axhline(THRESHOLD_mgL, color='0.4', ls='--', lw=1.4, label=f'threshold = {THRESHOLD_mgL:g} mg/L')
if t_first is not None:
    ax.axvline(t_first, color='steelblue', ls=':', lw=1.4)
    ax.text(t_first, peak * 0.5, f'  first exceedance\n  ~day {t_first:.0f}',
            color='steelblue', fontsize=9, va='center')
ax.set_xlabel('time since spill [d]'); ax.set_ylabel('concentration [mg/L]')
ax.set_title('Breakthrough at the extraction (monitoring) well')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

verdict = ("REACHES the well and EXCEEDS the threshold" if exceed.any()
           else "reaches the well but stays BELOW the threshold")
print(f"Peak {peak:.2f} mg/L at day {t_peak:.0f}.   Verdict: {verdict}.")
if t_first is not None:
    print(f"First exceedance of {THRESHOLD_mgL:g} mg/L at ~day {t_first:.0f}.")

---
## 4 — Does the model conserve mass?

Two sanity checks before trusting any number. **Mass balance:** the SRC mass we put in must equal what leaves at the well + boundaries + what is still stored, to within a tiny numerical imbalance. **Solubility:** because SRC sets a *mass rate*, the emergent source-cell concentration must stay **below the contaminant's solubility** — otherwise it implies separate-phase NAPL, which this dissolved-transport model cannot represent.

In [ ]:
print("Mass balance (from the GWT budget) [g]:")
for k, v in res.mass_balance.items():
    print(f"  {k:<30} {v:14.4g}")
print()
print(f"Solubility guardrail: emergent source concentration {res.emergent_C_mgL:.1f} mg/L "
      f"vs solubility {res.solubility_mgL:.0f} mg/L  ->  "
      f"{'PASS' if res.solubility_ok else 'FAIL'} (x{res.solubility_margin:.1f} margin)")

---
## 5 — What the grid supports — and the lateral question

The refined corridor resolves the **longitudinal** breakthrough (`Pe_L ≤ 2`), so **receptor arrival, peak, and threshold-exceedance timing are defensible**. But `Pe_T ≫ 2` everywhere: the plume's **lateral width** is set by numerical dispersion, not physics, and **no feasible grid** fixes it. So do **not** claim a contaminated-*area* or an exact concentration contour.

> **The right tool for the lateral / wellfield question is particle tracking.** A **capture zone** (MODFLOW 6 PRT) is *advective* and free of numerical dispersion — it answers "*which* area's water reaches the well," which the smeared concentration field cannot. That's a Step 8 application.

### Checkpoint 3 — defensible threshold claims

In [ ]:
create_multiple_choice('task_t04_checkpoint_3')

---
## Summary and handoff

You built and ran the spill→capture GWT model, read the breakthrough at the monitoring well, and confronted the core trade-off of transport modelling.

**What you're taking forward**

| Result | Trust it? |
|---|---|
| Receptor breakthrough / peak / arrival-time | ✅ resolved (`Pe_L ≤ 2` on the refined corridor) |
| Threshold-exceedance verdict at the well | ✅ defensible |
| Lateral plume width / contaminated area | ❌ numerical artefact (`Pe_T ≫ 2`) → use **particle tracking** (Step 8) |
| Mass balance | ✅ closes to a tiny imbalance |
| Source concentration vs solubility | ✅ checked |

**Key idea:** a **known-mass SRC pulse** gives a grid-independent, mass-conservative source; refine the corridor for the longitudinal breakthrough; and answer the lateral / wellfield question with **particle tracking**, not the smeared concentration field.

**Continue to:** [Step 5 — Calibration](./05t_calibration.ipynb) (a short bridge) → [Step 8 — Applications](./08t_model_application.ipynb).